# Dependencies


In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime


PATH = "../data/raw/listings.csv"
OUTPUT = "../reports/"

# Data


In [42]:
df = pd.read_csv(PATH)
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

Loaded 15,166 rows x 51 columns


In [43]:
schema_report = []
for col in df.columns:
    dtype = df[col].dtype
    n_unique = df[col].nunique()
    n_missing = df[col].isna().sum()
    pct_missing = 100 * n_missing / len(df)
    sample_df = df[col].dropna().head(3).to_list()

    schema_report.append(
        {
            "column": col,
            "dtype": dtype,
            "unique_count": n_unique,
            "missing_count": n_missing,
            "pct_missing": f"{pct_missing:.2f}%",
            "sample_values": str(sample_df)[:60],
        }
    )

schema_df = pd.DataFrame(schema_report)
# print(schema_df.to_string(index=False))
schema_df.to_csv(f"{OUTPUT}schema_inspection.csv", index=False)

In [44]:
schema_df.head()

,column,dtype,unique_count,missing_count,pct_missing,sample_values
0,Unnamed: 0,int64,15166,0,0.00%,"[0, 1, 2]"
1,url,object,15166,0,0.00%,['https://jiji.com.gh/kwabenya/houses-apartmen...
2,fetch_date,object,15166,0,0.00%,"['2025-12-25 20:38:06.320962', '2025-12-25 20:..."
3,title,object,8524,0,0.00%,"['4bdrm House in Acp, Kwabenya for rent', '2bd..."
4,location,object,312,0,0.00%,"['Ga West Municipal, Kwabenya', 'Dansoman', 'M..."


In [45]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_pct = (100 * missing_summary / len(df)).round(2)

missing_report = pd.DataFrame(
    {"missing_count": missing_summary, "missing_pct": missing_pct}
).query("missing_count > 0")

if len(missing_report) > 0:
    print("\nColumns with missing values:")
    print(missing_report)

    # Visualize missing patterns
    plt.figure(figsize=(10, 6))
    missing_report["missing_pct"].plot(kind="barh")
    plt.xlabel("Missing %")
    plt.title("Missing Data by Feature")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT}missing_values.png", dpi=150)
    plt.close()
    print("✓ Saved missing value visualization")
else:
    print("✓ No missing values found")



Columns with missing values:
                       missing_count  missing_pct
New Property                   15164        99.99
Housing Quality                15162        99.97
Listing by                     15162        99.97
Smoking                        15161        99.97
Parties                        15161        99.97
Broker Fee                     15161        99.97
Total Rooms                    15160        99.96
Parking Space                  15160        99.96
Service Charge                 15055        99.27
Facilities                     15040        99.17
Pets                           14747        97.24
Service Charge Covers          14191        93.57
Service Charge Fee             14027        92.49
Subtype                        13135        86.61
Caution Fee                    12153        80.13
Agency Fee                     10229        67.45
Minimum Rental Period           9633        63.52
Toilets                         4325        28.52
Estate Name         

In [46]:
price_col = (
    "price"
)
print(f"Using price column: '{price_col}'")

prices = df[price_col].dropna()

# Raw price statistics
price_stats = {
    "count": len(prices),
    "mean": prices.mean(),
    "median": prices.median(),
    "std": prices.std(),
    "min": prices.min(),
    "max": prices.max(),
    "q25": prices.quantile(0.25),
    "q75": prices.quantile(0.75),
    "iqr": prices.quantile(0.75) - prices.quantile(0.25),
}

print("\nRaw Price Statistics:")
for k, v in price_stats.items():
    print(f"  {k:10s}: {v:,.2f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")

# Log price statistics
log_prices = np.log(prices[prices > 0])
log_stats = {
    "count": len(log_prices),
    "mean": log_prices.mean(),
    "median": log_prices.median(),
    "std": log_prices.std(),
}

print("\nLog Price Statistics:")
for k, v in log_stats.items():
    print(f"  {k:10s}: {v:,.4f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")


Using price column: 'price'

Raw Price Statistics:
  count     : 15,166
  mean      : 253,096.12
  median    : 5,000.00
  std       : 16,810,017.54
  min       : 150.00
  max       : 1,655,500,000.00
  q25       : 2,500.00
  q75       : 15,000.00
  iqr       : 12,500.00

Log Price Statistics:
  count     : 15,166
  mean      : 8.6946
  median    : 8.5172
  std       : 1.3191


In [47]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Raw price histogram
axes[0, 0].hist(prices, bins=50, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Price")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Raw Price Distribution")

# Raw price boxplot
axes[0, 1].boxplot(prices, vert=True)
axes[0, 1].set_ylabel("Price")
axes[0, 1].set_title("Raw Price Boxplot")

# Log price histogram
axes[1, 0].hist(log_prices, bins=50, edgecolor="black", alpha=0.7, color="green")
axes[1, 0].set_xlabel("Log(Price)")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Log Price Distribution")

# Q-Q plot for log prices

stats.probplot(log_prices, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot (Log Price)")

plt.tight_layout()
plt.savefig(f"{OUTPUT}price_distributions.png", dpi=150)
plt.close()
print("✓ Saved price distribution visualizations")


✓ Saved price distribution visualizations


In [48]:
n_exact_dupes = df.duplicated().sum()
pct_exact_dupes = 100 * n_exact_dupes / len(df)
print(f"Exact duplicate rows: {n_exact_dupes:,} ({pct_exact_dupes:.2f}%)")

if "url" in df.columns or "id" in df.columns:
    id_col = "url" if "url" in df.columns else "id"
    n_id_dupes = df[id_col].duplicated().sum()
    pct_id_dupes = 100 * n_id_dupes / len(df)
    print(f"Duplicate {id_col}s: {n_id_dupes:,} ({pct_id_dupes:.2f}%)")

    # Show distribution of duplicate counts
    dupe_counts = df[id_col].value_counts()
    print(f"\nListings appearing multiple times: {(dupe_counts > 1).sum():,}")
    print(f"Max appearances: {dupe_counts.max()}")

    if (dupe_counts > 1).sum() > 0:
        plt.figure(figsize=(10, 6))
        dupe_counts[dupe_counts > 1].hist(bins=30, edgecolor="black")
        plt.xlabel("Number of Appearances")
        plt.ylabel("Number of Listings")
        plt.title("Distribution of Duplicate Listings")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT}duplicate_listings.png", dpi=150)
        plt.close()


Exact duplicate rows: 0 (0.00%)
Duplicate urls: 0 (0.00%)

Listings appearing multiple times: 0
Max appearances: 1


In [49]:
neighborhood_col = 'location'

if neighborhood_col:
    print(f"Using neighborhood column: '{neighborhood_col}'")

    # Listings per neighborhood
    nbhd_counts = df[neighborhood_col].value_counts()
    print(f"\nTotal neighborhoods: {len(nbhd_counts)}")
    print(f"Neighborhoods with <10 listings: {(nbhd_counts < 10).sum()}")
    print(f"Neighborhoods with <50 listings: {(nbhd_counts < 50).sum()}")
    print(f"Neighborhoods with <100 listings: {(nbhd_counts < 100).sum()}")

    print("\nTop 10 neighborhoods by listing count:")
    print(nbhd_counts.head(10))

    print("\nBottom 10 neighborhoods by listing count:")
    print(nbhd_counts.tail(10))

    # Save full neighborhood summary
    nbhd_summary = pd.DataFrame(
        {
            "neighborhood": nbhd_counts.index,
            "listing_count": nbhd_counts.values,
            "pct_of_total": 100 * nbhd_counts.values / len(df),
        }
    )
    nbhd_summary.to_csv(f"{OUTPUT}neighborhood_summary.csv", index=False)

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Top 20 neighborhoods
    nbhd_counts.head(20).plot(kind="barh", ax=axes[0])
    axes[0].set_xlabel("Number of Listings")
    axes[0].set_title("Top 20 Neighborhoods")

    # Distribution of listing counts
    axes[1].hist(nbhd_counts, bins=50, edgecolor="black")
    axes[1].set_xlabel("Listings per Neighborhood")
    axes[1].set_ylabel("Number of Neighborhoods")
    axes[1].set_title("Distribution of Listings Across Neighborhoods")
    axes[1].axvline(50, color="red", linestyle="--", label="50 listings")
    axes[1].axvline(100, color="orange", linestyle="--", label="100 listings")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"{OUTPUT}neighborhood_analysis.png", dpi=150)
    plt.close()

    # Monthly analysis if date available
    # if "date" in df.columns:
    #     df["month"] = pd.to_datetime(df["date"]).dt.to_period("M")
    #     monthly_nbhd = (
    #         df.groupby([neighborhood_col, "month"]).size().reset_index(name="count")
    #     )

    #     # Neighborhoods per month
    #     nbhd_per_month = monthly_nbhd.groupby("month")[neighborhood_col].nunique()
    #     print(f"\nAverage neighborhoods active per month: {nbhd_per_month.mean():.1f}")

    #     # Listings per neighborhood per month
    #     avg_listings_per_nbhd_month = monthly_nbhd["count"].mean()
    #     print(
    #         f"Average listings per neighborhood per month: {avg_listings_per_nbhd_month:.1f}"
    #     )

    #     monthly_nbhd.to_csv(f"{OUTPUT_DIR}neighborhood_monthly.csv", index=False)

else:
    print("⚠ No neighborhood column found")


Using neighborhood column: 'location'

Total neighborhoods: 312
Neighborhoods with <10 listings: 177
Neighborhoods with <50 listings: 251
Neighborhoods with <100 listings: 276

Top 10 neighborhoods by listing count:
location
East Legon                  1521
Spintex                      863
Accra Metropolitan           737
Tema Metropolitan            605
Airport Residential Area     603
Teshie, Tseaddo              594
Adenta                       576
Oyarifa                      431
Adjiriganor                  411
Teshie                       330
Name: count, dtype: int64

Bottom 10 neighborhoods by listing count:
location
Accra New Town                     1
Teshie, Lascala                    1
Dansoman, Sharp Curve              1
Kwashieman, A-Lang / Kwashieman    1
Kasoa, Brigade / Kasoa             1
Banana Inn, St Mary's Area         1
Bubuashie, Ola Balm                1
Kaneshie, Awudome Estate           1
Kokomlemle, ATTC Area              1
Dodowa, Dodowa Market             

In [50]:
summary = f"""
# Phase 1: Data Reality Check Summary
**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Dataset Overview
- **Total rows:** {len(df):,}
- **Total columns:** {len(df.columns)}
- **Date range:** {df["date"].min() if "date" in df.columns else "N/A"} to {df["date"].max() if "date" in df.columns else "N/A"}

## Data Quality Issues
- **Columns with missing data:** {len(missing_report) if len(missing_report) > 0 else 0}
- **Highest missing %:** {missing_report["missing_pct"].max() if len(missing_report) > 0 else 0}%
- **Exact duplicate rows:** {n_exact_dupes:,} ({pct_exact_dupes:.2f}%)

## Price Distribution
- **Mean price:** ${price_stats["mean"]:,.2f}
- **Median price:** ${price_stats["median"]:,.2f}
- **Price range:** ${price_stats["min"]:,.2f} - ${price_stats["max"]:,.2f}
- **Log price std:** {log_stats["std"]:.4f}

## Spatial Coverage
- **Total neighborhoods:** {len(nbhd_counts) if neighborhood_col else "N/A"}
- **Data-poor neighborhoods (<50 listings):** {(nbhd_counts < 50).sum() if neighborhood_col else "N/A"}
- **Data-poor neighborhoods (<100 listings):** {(nbhd_counts < 100).sum() if neighborhood_col else "N/A"}

## Key Questions to Answer

### 1. Which features are unusable?
- Review `schema_inspection.csv` for high missing % columns
- Review `missing_values.png` visualization
- **Decision threshold:** Consider dropping features with >50% missing

### 2. Which neighborhoods are data-poor?
- Review `neighborhood_summary.csv`
- Review `neighborhood_analysis.png`
- **Decision threshold:** Flag neighborhoods with <50-100 listings for shrinkage

### 3. What is the minimum sample size for modeling?
- Based on neighborhood analysis: ____ listings minimum
- Based on temporal coverage: ____ months minimum
- **Recommendation:** Set explicit thresholds before Phase 2

### 4. Is shrinkage/regularization justified?
- **Yes** if many data-poor neighborhoods exist
- **Yes** if high feature-to-sample ratio in small neighborhoods


## Files Generated
- `schema_inspection.csv`
- `missing_values.png`
- `missing_pattern.png`
- `price_distributions.png`
- `neighborhood_summary.csv`
- `neighborhood_analysis.png`
- `neighborhood_monthly.csv` (if date available)
- `duplicate_listings.png` (if duplicates found)
"""

# Save summary
with open(f"{OUTPUT}PHASE1_SUMMARY.md", "w") as f:
    f.write(summary)
